In [1]:
import pandas as pd
import numpy as np
from mlforecast import MLForecast
import sys
import os
sys.path.append(os.path.abspath("../.."))
from mlforecast.lag_transforms import RollingMean, RollingStd
import lightgbm as lgb
import holidays
from tinyshift.modelling import FirstStageForecasterEvaluator, TwoStageForecasterEvaluator, TwoStageForecasterWrapper

In [2]:
def generate_m5_simulated_data(n_stores=3, n_skus=5, start_date="2023-01-01", days=365, seed=42):

    np.random.seed(seed)
    dates = pd.date_range(start=start_date, periods=days, freq="D")
    n_days = len(dates)
    
    data_list = []

    for store_id in range(1, n_stores + 1):
        store_name = f"STORE_{store_id:02d}"
        for sku_id in range(1, n_skus + 1):
            item_id = f"FOODS_1_{sku_id:03d}"
            
            base_demand = np.random.uniform(0.5, 5.0)
            
            dow_factor = np.tile([0.8, 0.85, 0.9, 0.95, 1.1, 1.4, 1.3], int(np.ceil(n_days/7)))[:n_days]
            
            base_price = np.random.uniform(2.0, 15.0)
            prices = base_price * np.random.choice([1.0, 0.85, 0.70], size=n_days, p=[0.8, 0.15, 0.05])
            price_elasticity = np.exp(-0.15 * (prices - base_price))
            
            event_indices = np.random.choice(n_days, size=12, replace=False)
            event_impact = np.ones(n_days)
            event_impact[event_indices] = np.random.uniform(1.3, 2.2, size=12)
            
            lambda_t = base_demand * dow_factor * price_elasticity * event_impact
            
            sales = np.random.poisson(lambda_t)
            
            for d_idx, d_date in enumerate(dates):
                data_list.append({
                    'date': d_date,
                    'store_id': store_name,
                    'item_id': item_id,
                    'sales': sales[d_idx],
                    'sell_price': round(prices[d_idx], 2),
                    'is_event': 1 if d_idx in event_indices else 0,
                })

    return pd.DataFrame(data_list)

df_raw = generate_m5_simulated_data(n_stores=3, n_skus=5, days=365)

df_raw['unique_id'] = df_raw['store_id'] + '_' + df_raw['item_id']

df_nixtla = df_raw.rename(columns={
    'date': 'ds',
    'sales': 'y'
})

In [3]:
def temporal_split_by_horizon(df, time_col='ds', horizon_days=28):
    df = df.sort_values(time_col)
    max_date = df[time_col].max()
    cutoff_date = max_date - pd.Timedelta(days=horizon_days)
    
    df_train = df[df[time_col] <= cutoff_date].copy()
    df_test = df[df[time_col] > cutoff_date].copy()
    
    return df_train, df_test, cutoff_date

In [4]:
us_holidays = holidays.US(years=range(2022, 2027))

_extended_holiday_dates = set()
for h_date in us_holidays.keys():
    h_timestamp = pd.Timestamp(h_date)
    for offset in range(-2, 1): 
        _extended_holiday_dates.add((h_timestamp + pd.Timedelta(days=offset)).date())


def is_holiday_window(dates) -> pd.Series:
    dates_series = pd.Series(dates)
    return dates_series.dt.date.isin(_extended_holiday_dates).astype(int)

def add_relative_price(df: pd.DataFrame, id_col: str = 'unique_id', price_col: str = 'sell_price') -> pd.DataFrame:
    df = df.copy()
    
    mean_price_per_sku = df.groupby(id_col)[price_col].transform('mean')
    
    df['relative_price'] = df[price_col] / (mean_price_per_sku + 1e-6)
    
    return df

df_nixtla['is_holiday_window'] = is_holiday_window(df_nixtla['ds'])
df_nixtla = add_relative_price(df_nixtla)
df_train, df_test, cutoff = temporal_split_by_horizon(df_nixtla, horizon_days=28)

In [5]:
fcst = MLForecast(
    models={
        'two_stage': lgb.LGBMRegressor(
            objective='poisson',
            metric='poisson',
            n_estimators=100,
            learning_rate=1e-3,
            random_state=42,
            verbosity=-1,
        )
    },
    freq='D',
    lags=[7, 14, 28],
    lag_transforms={
        1: [RollingMean(window_size=7), RollingStd(window_size=7)],
    },
    date_features=['dayofweek', 'month', 'dayofyear']
)

In [6]:
tsf = TwoStageForecasterWrapper(fcst)
tsf.fit(df_train[["ds", "y", "sell_price", 'relative_price', "is_event", "is_holiday_window", "unique_id"]], h=28, n_windows=10, refit=True, gamma=None)

,fcst,MLForecast(mo...num_threads=1)


In [7]:
df_test[["ds", "sell_price", "is_event", "is_holiday_window", "unique_id"]].groupby("unique_id")["ds"].nunique()

unique_id
STORE_01_FOODS_1_001    28
STORE_01_FOODS_1_002    28
STORE_01_FOODS_1_003    28
STORE_01_FOODS_1_004    28
STORE_01_FOODS_1_005    28
STORE_02_FOODS_1_001    28
STORE_02_FOODS_1_002    28
STORE_02_FOODS_1_003    28
STORE_02_FOODS_1_004    28
STORE_02_FOODS_1_005    28
STORE_03_FOODS_1_001    28
STORE_03_FOODS_1_002    28
STORE_03_FOODS_1_003    28
STORE_03_FOODS_1_004    28
STORE_03_FOODS_1_005    28
Name: ds, dtype: int64

In [8]:
df_res = tsf.pmf(h=28, X_df=df_test[["ds", "sell_price", "relative_price", "is_event", "is_holiday_window", "unique_id"]], max_k=4)
df_res.loc[:, "y"] = df_test["y"].values

In [9]:
df_res

,unique_id,ds,lambda_t,r_dispersion,P(Y=0),P(Y=1),P(Y=2),P(Y=3),P(Y=4),P(Y>4),y
0,STORE_01_FOODS_1_001,2023-12-04,2.546518,10.976347,0.101257,0.209295,0.236011,0.192239,0.126489,0.134710,1
1,STORE_01_FOODS_1_001,2023-12-05,2.549491,10.976347,0.101013,0.208989,0.235888,0.192321,0.126663,0.135126,5
2,STORE_01_FOODS_1_001,2023-12-06,2.566212,10.976347,0.099652,0.207270,0.235192,0.192772,0.127635,0.137479,1
3,STORE_01_FOODS_1_001,2023-12-07,2.563597,10.976347,0.099863,0.207538,0.235302,0.192703,0.127484,0.137110,3
4,STORE_01_FOODS_1_001,2023-12-08,2.668816,10.976347,0.091729,0.196926,0.230642,0.195123,0.133347,0.152233,3
...,...,...,...,...,...,...,...,...,...,...,...
415,STORE_03_FOODS_1_005,2023-12-27,2.508944,3.456712,0.151627,0.220431,0.206581,0.158027,0.107279,0.156055,4
416,STORE_03_FOODS_1_005,2023-12-28,2.508944,3.456712,0.151627,0.220431,0.206581,0.158027,0.107279,0.156055,0
417,STORE_03_FOODS_1_005,2023-12-29,2.636530,3.456712,0.140932,0.210793,0.203247,0.159963,0.111726,0.173339,2
418,STORE_03_FOODS_1_005,2023-12-30,2.636530,3.456712,0.140932,0.210793,0.203247,0.159963,0.111726,0.173339,2


In [10]:
df_res = tsf.marginal_benefit(h=28, X_df=df_test[["ds", "sell_price", "relative_price", "is_event", "is_holiday_window", "unique_id"]], underage_cost=200, overage_cost=100, max_k=4)
df_res.loc[:, "y"] = df_test["y"].values
df_res

,unique_id,ds,lambda_t,r_dispersion,MB(k=0),MB(k=1),MB(k=2),MB(k=3),MB(k=4),y
0,STORE_01_FOODS_1_001,2023-12-04,2.546518,10.976347,200.0,169.623015,106.834449,36.031229,-21.640407,1
1,STORE_01_FOODS_1_001,2023-12-05,2.549491,10.976347,200.0,169.696214,106.999613,36.233122,-21.463201,5
2,STORE_01_FOODS_1_001,2023-12-06,2.566212,10.976347,200.0,170.104389,107.923481,37.365974,-20.465760,1
3,STORE_01_FOODS_1_001,2023-12-07,2.563597,10.976347,200.0,170.040959,107.779590,37.189140,-20.621806,3
4,STORE_01_FOODS_1_001,2023-12-08,2.668816,10.976347,200.0,172.481351,113.403481,44.210971,-14.325989,3
...,...,...,...,...,...,...,...,...,...,...
415,STORE_03_FOODS_1_005,2023-12-27,2.508944,3.456712,200.0,154.511902,88.382631,26.408412,-20.999803,4
416,STORE_03_FOODS_1_005,2023-12-28,2.508944,3.456712,200.0,154.511902,88.382631,26.408412,-20.999803,0
417,STORE_03_FOODS_1_005,2023-12-29,2.636530,3.456712,200.0,157.720515,94.482645,33.508440,-14.480401,2
418,STORE_03_FOODS_1_005,2023-12-30,2.636530,3.456712,200.0,157.720515,94.482645,33.508440,-14.480401,2


In [11]:
df_res = tsf.predict(h=28, X_df=df_test[["ds", "sell_price", "relative_price", "is_event", "is_holiday_window", "unique_id"]], quantiles=(0.05, 0.50, 0.95, 0.99))
df_res.loc[:, "y"] = df_test["y"].values

In [12]:
# The evaluator expects genuinely out-of-sample predictions. df_res uses the held-out test period.
first_stage_summary = FirstStageForecasterEvaluator.evaluate(
    df_res,
    id_col="unique_id",
)
first_stage_summary

,Metrics
WAPE,73.7504
PBias,3.9200
Score,77.6697
Forecast Instability,0.9888
False Demand on Zero-Days (Avg Pred),2.5555
Peak Demand Deviation (%),-16.8800


### First-stage conditional-mean diagnostics

Poisson Deviance is the primary proper loss for the non-negative conditional mean; lower is better. RMSE measures point accuracy, while PBias should be close to zero. Macro metrics give every series equal weight. The diagnostics conditioned on observed zero/positive demand are operational summaries, not tests of conditional-mean calibration.

In [13]:
calibration = FirstStageForecasterEvaluator.calibration_table(
    df_res,
    n_bins=10,
)
calibration

,Calibration Bin,Count,Mean_Prediction,Mean_Observed,Mean_Residual
0,"(2.463, 2.509]",110,2.505087,2.509091,0.004004
1,"(2.509, 2.536]",34,2.535819,2.441176,-0.094643
2,"(2.536, 2.549]",89,2.548957,2.808989,0.260032
3,"(2.549, 2.562]",77,2.561609,2.792208,0.230599
4,"(2.562, 2.564]",27,2.563391,1.703704,-0.859687
5,"(2.564, 2.636]",41,2.587324,1.682927,-0.904397
6,"(2.636, 2.803]",42,2.649452,2.214286,-0.435166


In a calibrated model, `Mean_Observed` is close to `Mean_Prediction` and `Mean_Residual` is close to zero in every bin. Persistent positive residuals indicate underforecasting; negative residuals indicate overforecasting.

In [14]:
TwoStageForecasterEvaluator.evaluate(df_res, quantiles=(0.05, 0.50, 0.95, 0.99))

,Pinball Loss,Target Coverage,Empirical Coverage,Coverage Gap
q_5,0.1229,0.05,0.2000,0.1500
q_50,0.8833,0.50,0.5738,0.0738
q_95,0.3312,0.95,0.9452,-0.0048
q_99,0.1135,0.99,0.9857,-0.0043


In [15]:
import joblib

model_path = "tsf.joblib"

# Save the fitted wrapper (works the same for mode="local" or mode="global").
joblib.dump(tsf, model_path)

# Load it back into a new object.
loaded_forecast = joblib.load(model_path)

In [16]:
loaded_forecast.predict(h=12, X_df=df_test[["ds", "sell_price", "relative_price", "is_event", "is_holiday_window", "unique_id"]])

,unique_id,ds,lambda_t,r_dispersion,q_5,q_50,q_95
0,STORE_01_FOODS_1_001,2023-12-04,2.546518,10.976347,0,2,6
1,STORE_01_FOODS_1_001,2023-12-05,2.549491,10.976347,0,2,6
2,STORE_01_FOODS_1_001,2023-12-06,2.566212,10.976347,0,2,6
3,STORE_01_FOODS_1_001,2023-12-07,2.563597,10.976347,0,2,6
4,STORE_01_FOODS_1_001,2023-12-08,2.668816,10.976347,0,2,6
...,...,...,...,...,...,...,...
175,STORE_03_FOODS_1_005,2023-12-11,2.562900,3.456712,0,2,7
176,STORE_03_FOODS_1_005,2023-12-12,2.562900,3.456712,0,2,7
177,STORE_03_FOODS_1_005,2023-12-13,2.549491,3.456712,0,2,7
178,STORE_03_FOODS_1_005,2023-12-14,2.563597,3.456712,0,2,7
